# 09 - DBSCAN and Anomaly Analysis
### Credit Portfolio Segmentation and Risk Pattern Analysis

---

## Detecting Risk Anomalies with DBSCAN
K-Means and Hierarchical Clustering assign every applicant to a group, even when an application is irregular or abnormal.
**DBSCAN (Density-Based Spatial Clustering)** behaves differently:
- It looks for dense zones of typical borrowers.
- Isolated applicants who sit in sparse feature space are labeled as **-1 (Anomalies)**.
- In credit portfolios, these outliers often point to extreme leverage, severe debt strain, or potential application irregularities.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA

X_scaled = pd.read_csv("../data/processed/scaled_features.csv")
df = pd.read_csv("../data/processed/cleaned_loans_dataset.csv")

print(f"Screening {len(X_scaled):,} borrowers for density anomalies.")


---
## 1. Running DBSCAN
- `eps=1.5`: Distance radius to search for neighbors.
- `min_samples=15`: Minimum neighbors needed to form a core cluster.


In [ ]:
dbscan = DBSCAN(eps=1.5, min_samples=15)
df['DBSCAN_Cluster'] = dbscan.fit_predict(X_scaled)

df['Is_Anomaly'] = df['DBSCAN_Cluster'] == -1

print("DBSCAN Breakdown:")
print(df['Is_Anomaly'].value_counts())
anomaly_rate = (df['Is_Anomaly'].sum() / len(df)) * 100
print(f"\nAnomaly Share: {anomaly_rate:.2f}% of the portfolio.")


---
## 2. Examining What Distinguishes Anomalies
Let's compare normal borrowers with anomalies across our main financial metrics.


In [ ]:
anomaly_comparison = df.groupby('Is_Anomaly')[
    ['annual_income', 'dti', 'loan_amount', 'fico_score', 'Default']
].mean()

anomaly_comparison['Default_Rate_%'] = anomaly_comparison['Default'] * 100
anomaly_comparison.round(2)


---
## 3. Visualizing Anomalies in PCA Space


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]

plt.figure(figsize=(9, 6))
sns.scatterplot(
    x='PC1', y='PC2', 
    hue='Is_Anomaly', 
    data=df.sample(2500, random_state=42), 
    palette={False: '#3498db', True: '#e74c3c'}, 
    alpha=0.8
)
plt.title("Portfolio Anomalies Detected via DBSCAN (Red = Outliers)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.tight_layout()
plt.savefig("../outputs/figures/dbscan_anomalies.png", dpi=300)
plt.show()
